In [13]:
# import pandas
import pandas as pd

In [14]:
credit_fraud_data = pd.read_csv("../data/credit_card_fraud_10k.csv")

credit_fraud_data.head()

,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
0,1,84.47,22,Electronics,0,0,66,3,40,0
1,2,541.82,3,Travel,1,0,87,1,64,0
2,3,237.01,17,Grocery,0,0,49,1,61,0
3,4,164.33,4,Grocery,0,1,72,3,34,0
4,5,30.53,15,Food,0,0,79,0,44,0


In [15]:
print (credit_fraud_data.shape)

(10000, 10)


In [16]:
print (credit_fraud_data.dtypes)

transaction_id           int64
amount                 float64
transaction_hour         int64
merchant_category          str
foreign_transaction      int64
location_mismatch        int64
device_trust_score       int64
velocity_last_24h        int64
cardholder_age           int64
is_fraud                 int64
dtype: object


In [17]:
print (credit_fraud_data.isnull().sum())

transaction_id         0
amount                 0
transaction_hour       0
merchant_category      0
foreign_transaction    0
location_mismatch      0
device_trust_score     0
velocity_last_24h      0
cardholder_age         0
is_fraud               0
dtype: int64


In [18]:
print (credit_fraud_data.duplicated().sum())

0


In [19]:
print(credit_fraud_data.describe())

       transaction_id        amount  transaction_hour  foreign_transaction  \
count     10000.00000  10000.000000      10000.000000         10000.000000   
mean       5000.50000    175.949849         11.593300             0.097800   
std        2886.89568    175.392827          6.922708             0.297059   
min           1.00000      0.000000          0.000000             0.000000   
25%        2500.75000     50.905000          6.000000             0.000000   
50%        5000.50000    122.095000         12.000000             0.000000   
75%        7500.25000    242.480000         18.000000             0.000000   
max       10000.00000   1471.040000         23.000000             1.000000   

       location_mismatch  device_trust_score  velocity_last_24h  \
count       10000.000000        10000.000000       10000.000000   
mean            0.085700           61.798900           2.008900   
std             0.279935           21.487053           1.432559   
min             0.000000     

In [20]:
print(credit_fraud_data.duplicated(subset="transaction_id").sum())

0


In [21]:
print(credit_fraud_data["is_fraud"].unique())

[0 1]


In [22]:
df_numeric = credit_fraud_data.select_dtypes(include=['number'])

print((df_numeric <= 0).sum())

transaction_id            0
amount                    1
transaction_hour        417
foreign_transaction    9022
location_mismatch      9143
device_trust_score        0
velocity_last_24h      1378
cardholder_age            0
is_fraud               9849
dtype: int64


In [23]:
print(credit_fraud_data["is_fraud"].value_counts())

is_fraud
0    9849
1     151
Name: count, dtype: int64


In [24]:
df_amount_zero = credit_fraud_data[credit_fraud_data["amount"]<= 0]

print (df_amount_zero["transaction_id"])

2902    2903
Name: transaction_id, dtype: int64


In [25]:
display(df_amount_zero)

,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
2902,2903,0.0,16,Travel,0,0,72,1,23,0


In [26]:
def risk_clasification (device_trust_score, velocity_last_24h):
    if device_trust_score < 40 and velocity_last_24h > 5:
        return "ALTO"
    elif device_trust_score < 40 or velocity_last_24h > 5:
        return "MEDIO"
    else:
        return "BAJO"

In [27]:
print(risk_clasification(39, 6))
print(risk_clasification(90, 2))
print(risk_clasification(39, 1))

ALTO
BAJO
MEDIO


In [31]:
credit_fraud_data['nivel_riesgo'] = credit_fraud_data.apply(
    lambda row: risk_clasification(row['device_trust_score'], row['velocity_last_24h']),
    axis = 1
)

print(credit_fraud_data.head())

print(credit_fraud_data['nivel_riesgo'].value_counts())

   transaction_id  amount  transaction_hour merchant_category  \
0               1   84.47                22       Electronics   
1               2  541.82                 3            Travel   
2               3  237.01                17           Grocery   
3               4  164.33                 4           Grocery   
4               5   30.53                15              Food   

   foreign_transaction  location_mismatch  device_trust_score  \
0                    0                  0                  66   
1                    1                  0                  87   
2                    0                  0                  49   
3                    0                  1                  72   
4                    0                  0                  79   

   velocity_last_24h  cardholder_age  is_fraud nivel_riesgo  
0                  3              40         0         BAJO  
1                  1              64         0         BAJO  
2                  1            

In [41]:
tabla_cruzada = pd.crosstab(
    index=credit_fraud_data["nivel_riesgo"],
    columns=credit_fraud_data["is_fraud"]
)

tabla_cruzada

is_fraud,0,1
nivel_riesgo,,
ALTO,31,11
BAJO,7880,25
MEDIO,1938,115
